In [ ]:
import json
from pathlib import Path
import os

import pandas as pd
from openai import OpenAI

# ========= 設定 =========
N_SAMPLES = 10  # ここを変えればサンプル数を指定できる

CAPTION_MODEL = "gpt-4o-mini"
QA_MODEL = "gpt-4o-mini"

META_CSV = "cholec80_sampled_dataset.csv"  # さっき作ったメタデータ
API_KEY_ENV = "OPENAI_API_KEY"

# ========= OpenAI client =========
api_key = os.getenv(API_KEY_ENV)
if not api_key:
    raise ValueError("Set OPENAI_API_KEY in environment.")
client = OpenAI(api_key=api_key)

# ========= メタデータ読み込み =========
df_meta = pd.read_csv(META_CSV)

# 必要ならシャッフルして先頭N件などでもOK
# df_meta = df_meta.sample(frac=1, random_state=42).reset_index(drop=True)
df_meta_n = df_meta.head(N_SAMPLES).copy()

# ========= 列定義 =========
STRUCTURE_COLS = [
    "hepatic_vein",
    "liver_ligament",
    "black_background",
    "l_hook_electrocautery",
    "grasper",
    "fat",
    "abdominal_wall",
    "gi_tract",
    "blood",
    "connective_tissue",
    "liver",
    "gallbladder",
    "cystic_duct_yellow",
    "cystic_duct_white",
]

TOOL_COLS = ["Grasper", "Bipolar", "Hook", "Scissors", "Clipper", "Irrigator", "SpecimenBag"]


# ========= 1) Caption 用プロンプト生成 =========
def build_caption_prompt_from_row(row):
    """
    cholec80_full_dataset.csv の 1行 (pd.Series) から
    英語の構造化キャプション(JSON)を生成させるための user プロンプトを作る。
    """
    video = int(row["video"])
    frame = int(row["frame"])
    phase = row.get("phase", "Unknown")

    present_structures = [
        col for col in STRUCTURE_COLS
        if col in row and row[col] > 0
    ]

    present_tools = [
        col for col in TOOL_COLS
        if col in row and row[col] == 1
    ]

    def list2text(lst):
        return ", ".join(lst) if lst else "none"

    structures_text = list2text(present_structures)
    tools_text = list2text(present_tools)

    prompt = f"""You are an expert in describing laparoscopic cholecystectomy scenes.

Below is metadata for a single video frame from the Cholec80 dataset. 
Use only this metadata to infer what is likely visible in the frame and generate a structured English description.

[Metadata]
- Dataset: Cholec80
- Video ID: video{video:02d}
- Frame index: {frame}
- Surgical phase label: {phase}
- Segmentation classes present in this frame (column names from the dataset):
  {structures_text}
- Binary tool annotations for this frame (columns with value 1 are present):
  {tools_text}

[Important notes]
- You do NOT see the image itself; you only see the metadata above.
- The segmentation class names are technical feature names from the dataset.
- You should convert them into natural anatomical or surgical terms when appropriate.
  For example, “gallbladder”, “liver”, “hepatic vein”, “cystic duct”, “abdominal wall”, 
  “gastrointestinal tract”, “blood”, etc.
- For tools, use realistic instrument names such as “grasper”, “bipolar forceps”, “hook cautery”, 
  “scissors”, “clip applier”, “irrigator/suction”, “specimen retrieval bag”, etc.
- Use the surgical phase label to infer what is likely happening (e.g., preparation, dissection, clipping, etc.),
  but do not invent impossible details.

[Output format]
Return ONLY a valid JSON object, with no extra text, no explanation and no markdown.
Use exactly the following keys:

{{
  "caption": string,                 // 1–2 natural English sentences summarizing what is visible in this frame
  "phase_description": string,       // a short phrase describing the surgical phase context in plain English
  "anatomy_present": [string],       // list of anatomical structures or regions likely visible
  "tools_present": [string],         // list of surgical tools/instruments likely visible
  "focus_of_frame": string           // a concise description of what the surgeon is mainly doing or focusing on
}}

Requirements:
- The JSON must be syntactically valid.
- The values should be consistent with the metadata.
- If no tools are present, use an empty list [] for "tools_present".
- If you are unsure about some details, make a reasonable, conservative guess based on the metadata.

Now output ONLY the JSON object for this frame."""

    return prompt


def call_caption_llm(row):
    prompt = build_caption_prompt_from_row(row)
    resp = client.chat.completions.create(
        model=CAPTION_MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful medical AI assistant."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.5,
    )
    content = resp.choices[0].message.content.strip()
    return json.loads(content)


# ========= 2) Caption → QA 用プロンプト =========
def build_qa_prompt_from_caption(caption_json):
    """
    caption_json: call_caption_llm の出力(JSONをdictにしたもの)
      {
        "caption": ...,
        "phase_description": ...,
        "anatomy_present": [...],
        "tools_present": [...],
        "focus_of_frame": ...
      }
    から、3問×6択のMCQAを作らせるプロンプトを生成。
    """
    caption = caption_json.get("caption", "")
    phase_desc = caption_json.get("phase_description", "")
    anatomy_present = caption_json.get("anatomy_present", [])
    tools_present = caption_json.get("tools_present", [])
    focus = caption_json.get("focus_of_frame", "")

    anatomy_text = ", ".join(anatomy_present) if anatomy_present else "none"
    tools_text = ", ".join(tools_present) if tools_present else "none"

    prompt = f"""You are creating multiple-choice question–answer (MCQA) items for a surgical vision-language dataset.

You are given a structured English description of a laparoscopic cholecystectomy frame:

[Structured description]
- Caption: {caption}
- Phase description: {phase_desc}
- Anatomy present: {anatomy_text}
- Tools present: {tools_text}
- Focus of frame: {focus}

Using ONLY this information, create 3 multiple-choice QA pairs (MCQ), each with:
- One question that can be answered from the description above
- Exactly 6 answer options (A–F), with only ONE correct answer
- Reasonable but clearly incorrect distractors that do not contradict the description

The questions should:
- Be about the surgical scene (phase, anatomy, tools, focus of action, etc.)
- Be answerable using the description alone
- Avoid trivial “copy one word” style; aim for clinically meaningful comprehension

[Output format]
Return ONLY a valid JSON object with the following structure, no extra text and no markdown:

{{
  "qas": [
    {{
      "id": 1,
      "question": "string",
      "options": ["option A", "option B", "option C", "option D", "option E", "option F"],
      "correct_option_index": int,   // 0-based index (0..5)
      "rationale": "short explanation of why the correct option is correct"
    }},
    {{
      "id": 2,
      "question": "...",
      "options": [... 6 strings ...],
      "correct_option_index": int,
      "rationale": "..."
    }},
    {{
      "id": 3,
      "question": "...",
      "options": [... 6 strings ...],
      "correct_option_index": int,
      "rationale": "..."
    }}
  ]
}}

Requirements:
- Exactly 3 questions.
- Each question must have exactly 6 options.
- Use 0-based indices for "correct_option_index" (0 to 5).
- All content must be consistent with the description above.

Now output ONLY this JSON object."""
    return prompt


def call_qa_llm(caption_json):
    prompt = build_qa_prompt_from_caption(caption_json)
    resp = client.chat.completions.create(
        model=QA_MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful medical AI assistant."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.7,
    )
    content = resp.choices[0].message.content.strip()
    return json.loads(content)


# ========= 3) ループして cholec_caption / cholec_qa を作成 =========
caption_rows = []
qa_rows = []

for idx, row in df_meta_n.iterrows():
    print(f"Processing index {idx}, video{int(row['video']):02d}, frame {int(row['frame'])}...")

    try:
        # --- caption生成 ---
        cap_json = call_caption_llm(row)

        # cholec_caption 用に整形
        caption_rows.append({
            "mask_path": row["mask_path"],
            "video": int(row["video"]),
            "frame": int(row["frame"]),
            "phase": row.get("phase", "Unknown"),
            "caption": cap_json.get("caption", ""),
            "phase_description": cap_json.get("phase_description", ""),
            "anatomy_present": ";".join(cap_json.get("anatomy_present", [])),
            "tools_present": ";".join(cap_json.get("tools_present", [])),
            "focus_of_frame": cap_json.get("focus_of_frame", ""),
        })

        # --- QA生成 ---
        qa_json = call_qa_llm(cap_json)
        for qa in qa_json.get("qas", []):
            qa_rows.append({
                "mask_path": row["mask_path"],
                "video": int(row["video"]),
                "frame": int(row["frame"]),
                "phase": row.get("phase", "Unknown"),
                "qa_id": qa.get("id"),
                "question": qa.get("question", ""),
                "option_0": qa.get("options", [""] * 6)[0] if len(qa.get("options", [])) > 0 else "",
                "option_1": qa.get("options", [""] * 6)[1] if len(qa.get("options", [])) > 1 else "",
                "option_2": qa.get("options", [""] * 6)[2] if len(qa.get("options", [])) > 2 else "",
                "option_3": qa.get("options", [""] * 6)[3] if len(qa.get("options", [])) > 3 else "",
                "option_4": qa.get("options", [""] * 6)[4] if len(qa.get("options", [])) > 4 else "",
                "option_5": qa.get("options", [""] * 6)[5] if len(qa.get("options", [])) > 5 else "",
                "correct_option_index": qa.get("correct_option_index", -1),
                "rationale": qa.get("rationale", ""),
            })

    except Exception as e:
        print(f"Error at index {idx}: {e}")
        continue

# DataFrame 化
cholec_caption = pd.DataFrame(caption_rows)
cholec_qa = pd.DataFrame(qa_rows)

# ついでにCSVとしても保存
cholec_caption.to_csv("../output/cholec_caption.csv", index=False)
cholec_qa.to_csv("../output/cholec_qa.csv", index=False)

print("Done. cholec_caption & cholec_qa are ready.")